<a href="https://colab.research.google.com/github/pushpendra-saini-pks/CP_LAB/blob/main/CP_LAB3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## RGB to Gray

In [2]:
import math
import os
import time
import numpy as np
from numba import cuda
from PIL import Image

@cuda.jit
def rgb_to_gray_gpu(rgb_img, gray_img):
    x, y = cuda.grid(2)

    if x < rgb_img.shape[0] and y < rgb_img.shape[1]:
        r = rgb_img[x, y, 0]
        g = rgb_img[x, y, 1]
        b = rgb_img[x, y, 2]

        gray_img[x, y] = 0.299*r + 0.587*g + 0.114*b


def rgb_to_gray_cpu(rgb):
    height, width = rgb.shape[:2]
    gray = np.zeros((height, width), dtype=np.float32)

    for i in range(height):
        for j in range(width):
            r, g, b = rgb[i, j]
            gray[i, j] = 0.299*r + 0.587*g + 0.114*b

    return gray


if __name__ == "__main__":
    BASE_DIR = '/content'

    # The input image is directly in BASE_DIR, not in a subfolder "images".
    input_image_path = os.path.join(BASE_DIR, "input_1.jpg")

    # Create the 'images' subdirectory for outputs if it doesn't exist
    output_images_dir = os.path.join(BASE_DIR, "images")
    os.makedirs(output_images_dir, exist_ok=True)

    output_cpu = os.path.join(output_images_dir, "gray_cpu.jpg")
    output_gpu = os.path.join(output_images_dir, "gray_gpu.jpg")

    img = Image.open(input_image_path).convert("RGB")
    rgb = np.array(img).astype(np.float32)

    height, width = rgb.shape[:2]


    start_cpu = time.perf_counter()

    gray_cpu = rgb_to_gray_cpu(rgb)

    cpu_time = time.perf_counter() - start_cpu

    Image.fromarray(np.clip(gray_cpu, 0, 255).astype(np.uint8)).save(output_cpu)


    gray_gpu = np.zeros((height, width), dtype=np.float32)

    start_gpu = time.perf_counter()

    d_rgb = cuda.to_device(rgb)
    d_gray = cuda.to_device(gray_gpu)

    threads_per_block = (16, 16)
    blocks_per_grid = (
        math.ceil(height / 16),
        math.ceil(width / 16),
    )

    rgb_to_gray_gpu[blocks_per_grid, threads_per_block](d_rgb, d_gray)

    cuda.synchronize()

    gray_result = d_gray.copy_to_host()

    gpu_time = time.perf_counter() - start_gpu

    Image.fromarray(np.clip(gray_result, 0, 255).astype(np.uint8)).save(output_gpu)

    print("\n===== CPU vs GPU Comparison =====")
    print(f"CPU Time : {cpu_time:.6f} seconds")
    print(f"GPU Time : {gpu_time:.6f} seconds")
    print(f"Speedup  : {cpu_time/gpu_time:.2f}x")

    print("\nImages saved:")
    print(f"CPU Output -> {output_cpu}")
    print(f"GPU Output -> {output_gpu}")


===== CPU vs GPU Comparison =====
CPU Time : 1.710636 seconds
GPU Time : 2.083152 seconds
Speedup  : 0.82x

Images saved:
CPU Output -> /content/images/gray_cpu.jpg
GPU Output -> /content/images/gray_gpu.jpg


## Single image processing

In [3]:
import math
import os
import numpy as np
from numba import cuda
from PIL import Image

@cuda.jit
def rgb_to_gray(rgb_img, gray_img):
    x, y = cuda.grid(2)

    if x < rgb_img.shape[0] and y < rgb_img.shape[1]:
        r = rgb_img[x, y, 0]
        g = rgb_img[x, y, 1]
        b = rgb_img[x, y, 2]

        gray = 0.299 * r + 0.587 * g + 0.114 * b
        gray_img[x, y] = gray


def process_single_image(input_path, output_path):

    img = Image.open(input_path).convert("RGB")
    rgb = np.array(img).astype(np.float32)

    height, width = rgb.shape[:2]
    gray = np.zeros((height, width), dtype=np.float32)

    d_rgb = cuda.to_device(rgb)
    d_gray = cuda.to_device(gray)

    threads_per_block = (16, 16)
    blocks_x = math.ceil(height / threads_per_block[0])
    blocks_y = math.ceil(width / threads_per_block[1])
    blocks_per_grid = (blocks_x, blocks_y)

    rgb_to_gray[blocks_per_grid, threads_per_block](d_rgb, d_gray)

    gray_result = d_gray.copy_to_host()

    gray_uint8 = np.clip(gray_result, 0, 255).astype(np.uint8)
    Image.fromarray(gray_uint8).save(output_path)

    print(f"Saved: {output_path}")


if __name__ == "__main__":

    BASE_DIR = '/content'


    # Input image paths
    image1 = os.path.join(BASE_DIR, "input_1.jpg")
    image3 = os.path.join(BASE_DIR, "input_3.jpg")
    image4 = os.path.join(BASE_DIR, "input_4.jpg")


    output1 = os.path.join(BASE_DIR, "images", "gray_output1.jpg")
    output3 = os.path.join(BASE_DIR, "images", "gray_output3.jpg")
    output4 = os.path.join(BASE_DIR, "images", "gray_output4.jpg")

    process_single_image(image1, output1)
    process_single_image(image3, output3)
    process_single_image(image4, output4)

    print("All images processed successfully!")

Saved: /content/images/gray_output1.jpg
Saved: /content/images/gray_output3.jpg
Saved: /content/images/gray_output4.jpg
All images processed successfully!
